In [2]:
"""
daemon_client.py
----------------
Client-side connector to ssh_daemon.py.

Sends encrypted credentials to the credential daemon over a raw TCP socket
on port 9922, using the same JSON message protocol the daemon expects.

Supports both LAN (direct IP) and WAN (domain/public IP) targets.
The Fernet key must be pre-shared with the daemon — this is what proves
the sender is authorised; the daemon will reject anything it cannot decrypt.

Protocol (newline-terminated JSON):
    Send:    { "type": "credential", "key_hash": "<sha256hex>", "encrypted": "<hex>" }
    Receive: { "status": "ok", "block_index": N, "block_hash": "<hex>" }
             { "status": "error", "reason": "..." }

    Ping:
    Send:    { "type": "ping" }
    Receive: { "status": "ok", "type": "pong" }

    Chain verify:
    Send:    { "type": "verify" }
    Receive: { "status": "ok", "chain_valid": true, "length": N }
"""

import json
import socket
import hashlib
import logging
from pathlib import Path
from cryptography.fernet import Fernet

log = logging.getLogger(__name__)

DEFAULT_PORT    = 9922
DEFAULT_TIMEOUT = 10      # seconds
MAX_RECV_BYTES  = 4096

In [3]:

# ── Fernet ────────────────────────────────────────────────────────────────

def load_fernet(key_path: str = "fernet.key") -> Fernet:
    """
    Load the shared Fernet key.
    Must match the key on the daemon machine — copy it via:
        scp /etc/credential-daemon/fernet.key user@sender:/path/to/fernet.key
    """
    path = Path(key_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Fernet key not found at {path}. "
            "Copy it from the daemon machine: "
            "scp /etc/credential-daemon/fernet.key user@this-machine:./fernet.key"
        )
    return Fernet(path.read_bytes())


In [37]:
import base64
import hashlib

testString = "hello apple"

digest = hashlib.sha256(testString.encode()).digest()  # 32 bytes
key = base64.urlsafe_b64encode(digest)

print(key)

load_fernet("./fernet.key")

In [61]:

# ── Low-level socket send ─────────────────────────────────────────────────

def _send_message(host: str, port: int, message: dict, timeout: int = DEFAULT_TIMEOUT) -> dict:
    """Open a TCP connection, send a JSON message, return the parsed response."""
    raw = (json.dumps(message) + "\n").encode()
    #
    #
    # return test response
    return {"status":"ok", 'block_index': 3, 'block_hash':'Jb3wYF0it9Of50enIQ5jvFM5kveiP_m5bkBWJ3kRs0Y='} # test return check, include block index
    #
    #
    #
    try:
        with socket.create_connection((host, port), timeout=timeout) as sock:
            sock.sendall(raw)
            response = sock.recv(MAX_RECV_BYTES)
        return json.loads(response.decode().strip())
    except ConnectionRefusedError:
        raise ConnectionRefusedError(
            f"Daemon not reachable at {host}:{port} — is credential-daemon running?"
        )
    except socket.timeout:
        raise TimeoutError(f"Daemon at {host}:{port} did not respond within {timeout}s")
    except json.JSONDecodeError as e:
        raise ValueError(f"Daemon returned invalid JSON: {e}")



In [62]:
# case transmits - reference session static elements
# ::ffff:127.0.0.1 - - [05/May/2026 10:18:42] code 400, message Bad HTTP/0.9 request type ('{"type":')
# ::ffff:127.0.0.1 - - [05/May/2026 10:18:42] "{"type": "verify"}" 400 -

# test = {"type": "verify"}
# _send_message(host="127.0.0.1",port=8000, message=test)

In [63]:

# ── Public API ────────────────────────────────────────────────────────────

def ping(host: str, port: int = DEFAULT_PORT) -> bool:
    """
    Check if the daemon is alive.
    Returns True if it responds with pong, False otherwise.
    """
    try:
        resp = _send_message(host, port, {"type": "ping"})
        alive = resp.get("type") == "pong"
        log.info(f"Ping {host}:{port} → {'alive' if alive else 'no pong'}")
        return alive
    except Exception as e:
        log.warning(f"Ping failed: {e}")
        return False



In [64]:
# Ping sends; needs response
# code 400, message Bad HTTP/0.9 request type ('{"type":')
# "{"type": "ping"}" 400 -
# Send:    { "type": "ping" }
# Receive: { "status": "ok", "type": "pong" }
# ping("127.0.0.1", 8000) # test call

In [65]:

def verify_chain(host: str, port: int = DEFAULT_PORT) -> dict:
    """
    Ask the daemon to verify its ledger chain integrity.
    Returns { chain_valid: bool, length: int }
    """
    resp = _send_message(host, port, {"type": "verify"})
    if resp.get("status") != "ok":
        raise RuntimeError(f"Chain verify failed: {resp.get('reason')}")
    log.info(f"Chain @ {host}: valid={resp['chain_valid']}, length={resp['length']}")
    return resp


In [66]:
# Sends build ledger communication protocols
# code 400, message Bad HTTP/0.9 request type ('{"type":')
# "{"type": "ping"}" 400 -
# verify_chain("127.0.0.1", 8000)

In [67]:

def send_to_daemon(
    credential: dict,
    host: str,
    fernet: Fernet,
    port: int = DEFAULT_PORT,
    timeout: int = DEFAULT_TIMEOUT,
) -> dict:
    """
    Encrypt a credential and deliver it to the daemon for storage.

    Args:
        credential: dict with at least a 'key' field
        host:       daemon machine IP or hostname (LAN or WAN)
        fernet:     shared Fernet instance (key must match daemon's key)
        port:       daemon port (default 9922)
        timeout:    socket timeout in seconds

    Returns:
        { "status": "ok", "block_index": N, "block_hash": "<hex>" }

    Raises:
        ValueError, ConnectionRefusedError, TimeoutError, RuntimeError
    """
    key_value = credential.get("key")
    if not key_value:
        raise ValueError("Credential missing 'key' field")

    # Encrypt
    plaintext = key_value.encode()
    encrypted = fernet.encrypt(plaintext)
    key_hash  = hashlib.sha256(plaintext).hexdigest()

    message = {
        "type":      "credential",
        "key_hash":  key_hash,
        "encrypted": encrypted.hex(),
    }

    log.info(f"Sending credential to daemon @ {host}:{port} ...")
    resp = _send_message(host, port, message, timeout=timeout)

    if resp.get("status") != "ok":
        raise RuntimeError(f"Daemon rejected credential: {resp.get('reason')}")

    log.info(
        f"Credential stored — block #{resp['block_index']}, "
        f"hash: {resp['block_hash'][:16]}..."
    )
    return resp
    

In [68]:

send_to_daemon(credential={'key':'test'}, host='127.0.0.1', fernet=load_fernet('./fernet.key'), port=8000)



{'status': 'ok',
 'block_index': 3,
 'block_hash': 'Jb3wYF0it9Of50enIQ5jvFM5kveiP_m5bkBWJ3kRs0Y='}